# CI-GCI: Causal-Interventional Grounding & Counterfactual Inpainting for Med-VQA
### End-to-End Autonomous Kaggle Execution Runner
This notebook clones the repository, links datasets, fine-tunes the model on SLAKE and VQA-RAD, evaluates benchmarks, generates proof plots, and prints publication-ready Markdown tables.

### Cell 1: Clone Repository & Install Dependencies

In [1]:
# Clone or pull latest code from GitHub
!if [ -d "CI-GCI" ]; then cd CI-GCI && git pull origin main; else git clone https://github.com/FaezehMillerAI/CI-GCI.git; fi
%cd CI-GCI

# Install required libraries
!pip install -q torch torchvision transformers scikit-learn matplotlib seaborn pandas tqdm Pillow pyyaml
import torch
print(f"PyTorch Version: {torch.__version__} | CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU Device: {torch.cuda.get_device_name(0)}")

Cloning into 'CI-GCI'...
remote: Enumerating objects: 517, done.
remote: Counting objects: 100% (517/517), done.
remote: Compressing objects: 100% (472/472), done.
remote: Total 517 (delta 88), reused 474 (delta 45), pack-reused 0 (from 0)
Receiving objects: 100% (517/517), 17.24 MiB | 27.24 MiB/s, done.
Resolving deltas: 100% (88/88), done.
/kaggle/working/CI-GCI
PyTorch Version: 2.10.0+cu128 | CUDA Available: True
GPU Device: Tesla T4


### Cell 2: Recursive Dataset Linker
Automatically discovers SLAKE, VQA-RAD, and MS-CXR in Kaggle inputs (`/kaggle/input/`) and links them to local `data/`.

In [4]:
import os
import glob

os.makedirs("data/slake", exist_ok=True)
os.makedirs("data/VQA-RAD", exist_ok=True)
os.makedirs("data/ms-cxr", exist_ok=True)

# Link SLAKE
slake_jsons = glob.glob("/kaggle/input/**/train.json", recursive=True)
if slake_jsons:
    slake_dir = os.path.dirname(slake_jsons[0])
    print(f"Found SLAKE dataset at: {slake_dir}")
    !ln -sf {slake_dir}/* data/slake/

# Link VQA-RAD
rad_jsons = glob.glob("/kaggle/input/**/*VQA_RAD Dataset Public.json", recursive=True)
if rad_jsons:
    rad_dir = os.path.dirname(rad_jsons[0])
    print(f"Found VQA-RAD dataset at: {rad_dir}")
    !ln -sf {rad_dir}/* data/VQA-RAD/

print("Linked Datasets in data/:", os.listdir("data"))

Found SLAKE dataset at: /kaggle/input/datasets/rounakmandal/slake-vqa/Slake1.0
Found VQA-RAD dataset at: /kaggle/input/datasets/shashankshekhar1205/vqa-rad-visual-question-answering-radiology
Linked Datasets in data/: ['ms-cxr', 'VQA-RAD', 'slake']


### Cell 3: Verify Pipeline Integrity

In [5]:
!PYTHONPATH=. python3 scripts/verify_pipeline.py

       CI-GCI AUTOMATED PIPELINE VERIFICATION     

[Step 1/5] Verifying Data Loader...
-> Successfully loaded dataset. Total samples: 4919
-> Data Loader Verification: PASS

[Step 2/5] Verifying Gaze-Guided ROI Locator (GGRL)...
-> Question: 'Does the picture contain liver?' mapped to: ['liver cancer', 'liver']
-> ROI Locator Verification: PASS

[Step 3/5] Verifying Counterfactual Inpainter (CFI)...
-> Background deviation outside mask: 0.000000
-> Counterfactual Inpainter Verification: PASS

[Step 4/5] Verifying Causal Contrastive Decoder (CCD)...
-> Mock Input: Original Logits [2.0, -1.0], Counterfactual Logits [0.0, 0.0]
-> Calculated ICE: [[ 2. -1.]]
-> Hallucination Risk Score: [[0.04742587 0.81757444]]
-> Calibrated Probs: [[0.99055547 0.00944457]]
-> Causal Contrastive Decoder Verification: PASS

[Step 5/5] Running End-to-End Integration Check...
-> End-to-End Integration Verification: PASS

   ALL PIPELINE CHECKS PASSED SUCCESSFULLY!       


### Cell 4: Fine-Tune CI-GCI Model on SLAKE (15 Epochs)

In [6]:
!PYTHONPATH=. python3 training/train_slake_vqa.py --dataset slake --epochs 15 --batch_size 16 --lr 5e-4 --device cuda

Starting SLAKE VQA fine-tuning on device: cuda
Loaded 1943 train and 422 val samples with 33 answer classes.
Loading real ViT Vision Encoder: google/vit-base-patch16-224-in21k...
config.json: 100%|█████████████████████████████| 502/502 [00:00<00:00, 3.19MB/s]
model.safetensors: 100%|█████████████████████| 346M/346M [00:04<00:00, 85.5MB/s]
Loading weights: 100%|█| 200/200 [00:00<00:00, 1493.53it/s, Materializing param=
Successfully loaded pre-trained ViT Vision Encoder!
Loading real PubMedBERT Text Encoder: microsoft/BiomedNLP-BiomedBERT-base-uncased-abstract-fulltext...
config.json: 100%|█████████████████████████████| 385/385 [00:00<00:00, 2.14MB/s]
tokenizer_config.json: 100%|██████████████████| 28.0/28.0 [00:00<00:00, 152kB/s]
vocab.txt: 226kB [00:00, 13.3MB/s]
pytorch_model.bin: 100%|██████████████████████| 440M/440M [00:03<00:00, 129MB/s]
Loading weights: 100%|█| 199/199 [00:00<00:00, 2723.11it/s, Materializing param=
BertModel LOAD REPORT from: microsoft/BiomedNLP-BiomedBERT-base-

### Cell 5: Fine-Tune CI-GCI Model on VQA-RAD (15 Epochs)

In [7]:
!PYTHONPATH=. python3 training/train_slake_vqa.py --dataset vqa_rad --epochs 15 --batch_size 16 --lr 5e-4 --device cuda

Starting VQA_RAD VQA fine-tuning on device: cuda
Loaded 1037 train and 260 val samples with 57 answer classes.
Loading real ViT Vision Encoder: google/vit-base-patch16-224-in21k...
Loading weights: 100%|█| 200/200 [00:00<00:00, 1708.09it/s, Materializing param=
Successfully loaded pre-trained ViT Vision Encoder!
Loading real PubMedBERT Text Encoder: microsoft/BiomedNLP-BiomedBERT-base-uncased-abstract-fulltext...
Loading weights: 100%|█| 199/199 [00:00<00:00, 2637.34it/s, Materializing param=
BertModel LOAD REPORT from: microsoft/BiomedNLP-BiomedBERT-base-uncased-abstract-fulltext
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.decoder.weight             | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weig

### Cell 6: Run Comparative Benchmarks (SLAKE & VQA-RAD)

In [15]:
!git pull origin main

remote: Enumerating objects: 11, done.
remote: Counting objects: 100% (11/11), done.
remote: Compressing objects: 100% (1/1), done.
remote: Total 6 (delta 5), reused 6 (delta 5), pack-reused 0 (from 0)
Unpacking objects: 100% (6/6), 910 bytes | 455.00 KiB/s, done.
From https://github.com/FaezehMillerAI/CI-GCI
 * branch            main       -> FETCH_HEAD
   279d39b..45e8023  main       -> origin/main
Updating 279d39b..45e8023
Fast-forward
 evaluation/eval_calibration_grounding.py | 13 +++++++++----
 scripts/benchmark_comparison.py          | 10 +++++++---
 2 files changed, 16 insertions(+), 7 deletions(-)


In [16]:
print("=== BENCHMARKING SLAKE ===")
!PYTHONPATH=. python3 scripts/benchmark_comparison.py --dataset slake --device cuda

print("\n=== BENCHMARKING VQA-RAD ===")
!PYTHONPATH=. python3 scripts/benchmark_comparison.py --dataset vqa_rad --device cuda

=== BENCHMARKING SLAKE ===
Running Comparative Benchmarking on 'SLAKE' using device: cuda
Loaded 416 evaluation samples.
Loading real ViT Vision Encoder: google/vit-base-patch16-224-in21k...
Loading weights: 100%|█| 200/200 [00:00<00:00, 1597.82it/s, Materializing param=
Successfully loaded pre-trained ViT Vision Encoder!
Loading real PubMedBERT Text Encoder: microsoft/BiomedNLP-BiomedBERT-base-uncased-abstract-fulltext...
Loading weights: 100%|█| 199/199 [00:00<00:00, 2400.45it/s, Materializing param=
BertModel LOAD REPORT from: microsoft/BiomedNLP-BiomedBERT-base-uncased-abstract-fulltext
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.seq_relationship.bias     

### Cell 7: Generate Proof Plots & Reliability Diagrams

In [20]:
!PYTHONPATH=. python3 scripts/generate_plots_and_proofs.py --device cuda

Loading real ViT Vision Encoder: google/vit-base-patch16-224-in21k...
Loading weights: 100%|█| 200/200 [00:00<00:00, 1697.02it/s, Materializing param=
Successfully loaded pre-trained ViT Vision Encoder!
Loading real PubMedBERT Text Encoder: microsoft/BiomedNLP-BiomedBERT-base-uncased-abstract-fulltext...
Loading weights: 100%|█| 199/199 [00:00<00:00, 2582.36it/s, Materializing param=
BertModel LOAD REPORT from: microsoft/BiomedNLP-BiomedBERT-base-uncased-abstract-fulltext
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.decoder.bia

### Cell 8: Generate & Display 7 Publication-Ready Markdown Tables

In [19]:
!PYTHONPATH=. python3 evaluation/result_table_generator.py

import os
tables_dir = "outputs/tables"
if os.path.exists(tables_dir):
    for fname in sorted(os.listdir(tables_dir)):
        if fname.endswith(".md"):
            fpath = os.path.join(tables_dir, fname)
            print("="*80)
            print(f" DISPLAYING TABLE: {fname.upper()}")
            print("="*80)
            with open(fpath, "r") as f:
                print(f.read())
            print("\n")

[Tables] Generating Main VQA comparison results table...
[Tables] Generating Reasoning Breakdown results table...
[Tables] Generating Hallucination Detection table...
[Tables] Generating Grounding & Explanation table...
[Tables] Generating Human Evaluation table...
[Tables] Generating Calibration & Abstention table...
[Tables] Generating Ablation Modules table...
[Tables] Success! Generated 7 publication-ready tables in outputs/tables
 DISPLAYING TABLE: ABLATION_1_MODULES.MD
### Table 7: Ablation study of core modules

| Setting         | QCG   | Verifier   | Consistency Head   | Refiner   | Abstention   |      Acc |    F1 |   BLEU-4 |   CIDEr |   Halluc. Rate ↓ |   Halluc. F1 |   AUROC |    ECE ↓ |
|:----------------|:------|:-----------|:-------------------|:----------|:-------------|---------:|------:|---------:|--------:|-----------------:|-------------:|--------:|---------:|
| Full model      | ✓     | ✓          | ✓                  | ✓         | ✓            | 0.810096 | 0.778 |